# Load the excel file and sample the rays (initial position and direction)

In [40]:
import matplotlib.pyplot as plt
import scipy as sp
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import uniform
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import pandas as pd
from scipy.stats import norm
import scipy.stats as stats 

## Load the Excel file

In [41]:
df_structure = pd.read_excel("omega60_beam_structure.xlsx")

## Import of helper functions

### Gaussian random generation of points in a window

In [42]:
def sample_window( t1, t2, l1=1., l2=1., center=np.array([ 0., 0., 0., ]), num_points=1000, dist = lambda num_points: sp.stats.norm(loc=0.0, scale=0.5).rvs(size=(num_points, 2)), ):
  """
    Generates a set of random starting points (sampling) within a 2D rectangular 
    laser window defined by vectors t1 and t2 in 3D space.
    """
  sample = np.zeros( (num_points, 2,) )
  k = 0
  while k < num_points:
    s = dist(1)
    if np.abs(s[0,0]) <= 1.0 and np.abs(s[0,1]) <= 1.0:
      sample[k,:] = s[0,:]
      k += 1
  # map to the window
  v1 = 0.5 * l1 * t1 / np.linalg.norm( t1 )
  v2 = 0.5 * l2 * t2 / np.linalg.norm( t2 )
  return center + sample[:,0,None] * v1 + sample[:,1,None] * v2

## Generating the rays initial position and direction

In [43]:
# 2. PARAMETERS
N_RAYS = 500
FOCUS_POINT = np.array([0, 0, 0])

# scale=0.15 determines the beam concentration (Gaussian profile)
dist_gauss = lambda n: sp.stats.norm(loc=0.0, scale=0.15).rvs(size=(n, 2))

all_rays = []

# Iterating through each beam defined in the Excel structure
for idx, row in df_structure.iterrows():
    
    
    ray_origins = sample_window(
        t1 = np.array([row['t1_x'], row['t1_y'], row['t1_z']]),
        t2 = np.array([row['t2_x'], row['t2_y'], row['t2_z']]),
        l1 = row['l1'],
        l2 = row['l2'],
        center = np.array([row['center_x'], row['center_y'], row['center_z']]),
        num_points = N_RAYS,
        dist = dist_gauss
    )
    
    # --- CALCULATING RAY DIRECTIONS ---
    for i in range(N_RAYS):
        origin = ray_origins[i]
        
        # Calculate the unit direction vector pointing from origin to focus (0,0,0)
        direction = FOCUS_POINT - origin
        u_ray = direction / np.linalg.norm(direction)
        
        # Storing all ray data (metadata + initial position + direction)
        all_rays.append({
            'port': row['port'],
            'origin_x': origin[0], 
            'origin_y': origin[1], 
            'origin_z': origin[2],
            'dir_x': u_ray[0], 
            'dir_y': u_ray[1], 
            'dir_z': u_ray[2]
        })

# Create the final DataFrame containing all individual rays
df_rays = pd.DataFrame(all_rays)